# BI Intern Assignment — D2C Analytics Challenge
SQL (Task 1), Python (Task 2), LLM dashboard (Task 3).

**Revenue definition used throughout:** `quantity × MRP × (1 − discount_pct/100)`, where `discount_pct` comes from `product_pricing` joined on **both** `product_name` and `channel`.

**Assumptions / notes**
- The cover sheet's submission line says "2 python tasks", but the tab-navigation row on the same sheet says "1 Python task", and the `🐍 Python Task` tab itself defines only **QP1** (checked for hidden rows/cells/comments — none found). I've answered QP1 as specified.
- `total_orders` = `COUNT(order_id)`, not `SUM(quantity)`.
- QS3 ("highest discount_pct") is read as the average discount rate actually applied per channel across the orders in this file (order-count-weighted), not revenue-weighted or the max discount tier available. I show both avg and max so the reasoning is visible.

## Setup: load the workbook
The workbook (`source.xlsx`) should sit in the **same folder as this notebook**. If yours is named differently, change `WORKBOOK_PATH` below.

In [1]:
import pandas as pd, sqlite3
from pathlib import Path

WORKBOOK_PATH = Path("source.xlsx")
if not WORKBOOK_PATH.exists():
    # fall back to any single .xlsx found alongside the notebook
    candidates = list(Path(".").glob("*.xlsx"))
    if len(candidates) == 1:
        WORKBOOK_PATH = candidates[0]
    else:
        raise FileNotFoundError("Put the assignment workbook next to this notebook (or set WORKBOOK_PATH).")

orders    = pd.read_excel(WORKBOOK_PATH, sheet_name=1, nrows=50)
customers = pd.read_excel(WORKBOOK_PATH, sheet_name=2, nrows=26)
catalog   = pd.read_excel(WORKBOOK_PATH, sheet_name=3, nrows=18)
pricing   = pd.read_excel(WORKBOOK_PATH, sheet_name=4, nrows=90)

con = sqlite3.connect(":memory:")
for name, df in [("orders",orders),("customers",customers),("product_catalog",catalog),("product_pricing",pricing)]:
    df.astype({c: str for c in df.select_dtypes("datetime").columns}).to_sql(name, con, index=False)

print("orders w/o customer      :", len(set(orders.customer_id) - set(customers.customer_id)))
print("orders w/o pricing match :", orders.merge(pricing, on=["product_name","channel"], how="left").discount_pct.isna().sum())
print("duplicate order_ids      :", orders.order_id.duplicated().sum())

orders w/o customer      : 0
orders w/o pricing match : 0
duplicate order_ids      : 0


## Task 1 — SQL
### QS1. Orders and average MRP by customer segment

In [2]:
q1 = """
SELECT c.customer_segment,
       COUNT(o.order_id)   AS total_orders,
       ROUND(AVG(o.MRP),2) AS avg_MRP
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_segment
ORDER BY total_orders DESC;
"""
print(pd.read_sql(q1, con).to_string(index=False))

customer_segment  total_orders  avg_MRP
            Gold            22  1327.64
          Silver            18  1282.11
          Bronze            10  1282.90


Gold is the biggest segment (22 orders) and also has the highest average MRP.

### QS2. Total revenue by product
Join on **both** `product_name` and `channel`, since the discount differs by channel.

In [3]:
q2 = """
SELECT o.product_name,
       ROUND(SUM(o.quantity * o.MRP * (1 - p.discount_pct/100.0)),2) AS total_revenue
FROM orders o
JOIN product_pricing p
  ON o.product_name = p.product_name AND o.channel = p.channel
GROUP BY o.product_name
ORDER BY total_revenue DESC;
"""
print(pd.read_sql(q2, con).to_string(index=False))

       product_name  total_revenue
    Vitamin C Serum       30730.35
Collagen Supplement       20429.18
        Conditioner       14298.75
        Ashwagandha       14100.00
        Night Cream       13540.80
        Moisturizer       13172.60
          Hair Mask       12145.20
         Vitamin D3       10751.58
           Eyeliner       10709.26
    Probiotic Blend       10605.76
            Omega 3        9450.54
           Hair Oil        7148.40
           Lipstick        4440.75
          Face Wash        3645.00
    Sunscreen SPF50        3633.84
        Scalp Serum        2512.75
      Shampoo 250ml        1908.72
         Foundation         133.38


Vitamin C Serum leads at about ₹30.7k, well ahead of Collagen Supplement (₹20.4k). Foundation is last (₹133), because its MRP is only ₹171.

### QS3. Which channel gave the highest discount_pct on the orders made?
Each order gets its discount via `(product_name, channel)`. I show the average and the maximum per channel — see the assumptions note above on why average is the headline answer.

In [4]:
q3 = """
SELECT o.channel,
       COUNT(*)                     AS orders,
       ROUND(AVG(p.discount_pct),2) AS avg_discount_pct,
       MAX(p.discount_pct)          AS max_discount_pct
FROM orders o
JOIN product_pricing p
  ON o.product_name = p.product_name AND o.channel = p.channel
GROUP BY o.channel
ORDER BY avg_discount_pct DESC;
"""
print(pd.read_sql(q3, con).to_string(index=False))

  channel  orders  avg_discount_pct  max_discount_pct
   Amazon       7             17.43                22
Instagram      11             11.36                15
  Website       8              9.25                12
 WhatsApp       9              7.11                10
      App      15              5.53                10


**Amazon** gives the highest discount, with an average of 17.4% and a max of 22%, on either measure. The App gives the lowest average (5.5%) but has the most orders (15).

## Task 2 — Python
### QP1. Category | Total Revenue | Orders | Avg Order Value
Avg Order Value = Total Revenue ÷ Orders.

In [5]:
df = (orders.merge(pricing, on=["product_name","channel"], how="left")
            .merge(catalog[["category","product_name"]], on=["category","product_name"], how="left"))
df["revenue"] = df.quantity * df.MRP * (1 - df.discount_pct/100)

cat = (df.groupby("category")
         .agg(**{"Total Revenue":("revenue","sum"), "Orders":("order_id","nunique")})
         .reset_index().rename(columns={"category":"Category"}))
cat["Avg Order Value"] = cat["Total Revenue"] / cat["Orders"]
cat = cat.sort_values("Total Revenue", ascending=False).round(2).reset_index(drop=True)
print(cat.to_string(index=False))
print("\nTotal:", round(cat["Total Revenue"].sum(),2), "| Orders:", cat.Orders.sum())

Category  Total Revenue  Orders  Avg Order Value
Wellness       65337.06      16          4083.57
Skincare       64722.59      15          4314.84
Haircare       38013.82      14          2715.27
  Makeup       15283.39       5          3056.68

Total: 183356.86 | Orders: 50


Wellness brings in the most revenue (₹65.3k), narrowly ahead of Skincare (₹64.7k), while Skincare has the highest average order value (₹4,315). Makeup is smallest.

## Task 3 — LLM Dashboard
### Initial prompt given to the LLM

```text
You are a senior front-end BI engineer. Build a single, self-contained HTML file 
that runs directly in any browser with no server, no build step, and no backend. 
Everything — data, styles, and logic — must be inline in one file.

CONTEXT
This is a D2C beauty and wellness brand. I need a channel + product performance 
dashboard for my manager.

DATA
Embed this as a JS const array called ANALYTICS_DATA: 50 order records, each with 
order_id, order_date, customer_id, city, state, category, product_name, quantity, 
channel, MRP, discount_pct, customer_segment, and a pre-computed revenue field.
[paste the actual rows or attach the file]

BUSINESS LOGIC
revenue = quantity × MRP × (1 - discount_pct / 100)
discount_pct depends on BOTH product_name and channel — the same product has a 
different discount on Amazon vs. the App vs. Instagram, etc. Compute revenue once 
per row before embedding the data; don't recompute the formula in multiple places 
in the JS.

KPI CARDS
Total Revenue, Total Orders, Avg Order Value, Total Units Sold, Avg Discount %.
These must update live as filters change.

CHARTS (use Chart.js via CDN)
- Channel: revenue by channel, revenue share (doughnut), orders by channel, avg 
  discount by channel
- Product: top 10 by revenue, units sold, orders, avg discount
- Category: revenue by category, orders by category, avg order value by category

FILTERS
- Channel (multi-select)
- Category (multi-select)
- Product (single select, populated from the data)
- Customer segment (single select: All/Gold/Silver/Bronze)
- Month range (from/to)
- A "Reset filters" button
One function should apply all filter state to the raw records; every KPI and chart 
reads from that same filtered set — no separate filtering logic per chart.

DESIGN
Clean, modern, dark-themed dashboard. Use a real font (Inter via Google Fonts is 
fine). Mobile-friendly layout.

ROBUSTNESS
If Chart.js fails to load (e.g. no internet), the page should still show the KPIs 
and a data table rather than breaking entirely. Escape any user-facing values that 
get inserted into HTML (product names, etc.) to avoid injection.
```

### Dashboard link
https://rohitftw.github.io/d2c-dashboard/